# Cookie Cats A/B Testi - 1. Deney Tanımı ve Veri Kontrolü

Cookie Cats bir mobil bulmaca oyunu. Oyuncular belli bir seviyede bir kapıya (gate) geliyor ve devam etmek için ya bekliyor ya da satın alma yapıyor. Bu deneyde kapının yeri test edilmiş:

- `gate_30`: kapı level 30'da (mevcut durum, kontrol grubu)
- `gate_40`: kapı level 40'ta (test grubu)

**Soru:** Kapıyı 40'a taşımak oyuncuların oyunda kalma oranını değiştiriyor mu?

**Metrikler:**
- Ana metrik: `retention_7` (oyuncu 7 gün sonra oyuna döndü mü)
- Yardımcı metrik: `retention_1` (1 gün sonra döndü mü)
- Kontrol metriği: `sum_gamerounds` (ilk 14 günde oynanan round sayısı)

**Hipotez:** H0: iki grup arasında fark yok, H1: fark var. Çift yönlü test, α = 0.05.

Ana metrik olarak retention_7'yi seçtim çünkü kapı level 30-40 civarında ve oyuncuların çoğu ilk gün bu seviyeye gelemiyor. Kapının etkisi daha çok 7. günde görülür.

In [1]:
import pandas as pd
from scipy.stats import chisquare

df = pd.read_csv("../data/cookie_cats.csv")
df.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 90189 entries, 0 to 90188
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   userid          90189 non-null  int64
 1   version         90189 non-null  str  
 2   sum_gamerounds  90189 non-null  int64
 3   retention_1     90189 non-null  bool 
 4   retention_7     90189 non-null  bool 
dtypes: bool(2), int64(2), str(1)
memory usage: 2.2 MB


In [3]:
print("Eksik değer:", df.isnull().sum().sum())
print("Tekrar eden userid:", df["userid"].duplicated().sum())

Eksik değer: 0
Tekrar eden userid: 0


## Grup büyüklükleri

Oyuncular rastgele 50/50 dağıtıldıysa iki grubun büyüklüğü yakın olmalı. Bunu ki-kare testi ile kontrol ettim (SRM - sample ratio mismatch kontrolü).

In [4]:
counts = df["version"].value_counts()
print(counts)
print(counts / counts.sum())

stat, p = chisquare(counts)
print(f"\nchi2 = {stat:.2f}, p = {p:.4f}")

version
gate_40    45489
gate_30    44700
Name: count, dtype: int64
version
gate_40    0.504374
gate_30    0.495626
Name: count, dtype: float64

chi2 = 6.90, p = 0.0086


p ≈ 0.009 çıktı, yani gruplar beklenenden daha dengesiz (%49.6 / %50.4). Bu, atamada veya veri kaydında bir sorun olabileceğini gösteriyor. Atamanın nasıl yapıldığını bilmediğim için nedenini bulamıyorum; analize devam ediyorum ama sonuçları yorumlarken bunu bir sınırlama olarak not ediyorum.

## Round sayısı ve aşırı değerler

In [5]:
df["sum_gamerounds"].describe()

count    90189.000000
mean        51.872457
std        195.050858
min          0.000000
25%          5.000000
50%         16.000000
75%         51.000000
max      49854.000000
Name: sum_gamerounds, dtype: float64

In [6]:
df.sort_values("sum_gamerounds", ascending=False).head()

,userid,version,sum_gamerounds,retention_1,retention_7
57702,6390605,gate_30,49854,False,True
7912,871500,gate_30,2961,True,True
29417,3271615,gate_40,2640,True,False
43671,4832608,gate_30,2438,True,True
48188,5346171,gate_40,2294,True,True


Bir oyuncu 49.854 round oynamış görünüyor. İkinci en yüksek değer 2.961. 14 günde 49.854 round günde yaklaşık 3.500 round demek, bu gerçekçi değil, büyük ihtimalle veri hatası. Bu tek kaydı bir sonraki notebook'ta çıkarıyorum.

Diğer yüksek değerleri silmiyorum. Round sayısı deneyden etkilenen bir değişken (kapının yeri oyuncunun kaç round oynadığını değiştirebilir). Buna göre filtreleme yaparsam iki grubu farklı şekilde kırpmış olurum ve en çok oynayan, kapıyı kesin gören oyuncuları atmış olurum.

In [7]:
zero = df[df["sum_gamerounds"] == 0]
print("Hiç oynamayan oyuncu:", len(zero))
print(zero["version"].value_counts())

Hiç oynamayan oyuncu: 3994
version
gate_40    2057
gate_30    1937
Name: count, dtype: int64


3.994 oyuncu (%4.4) oyunu yükleyip hiç oynamamış. Bunlar kapıya ulaşmadığı için etkilenmiyor ama rastgele atanmışlar ve iki grupta sayıları benzer. Atanan herkesi analize dahil etmek (intent-to-treat) için bunları tutuyorum.

## Özet
- Eksik değer ve tekrar eden oyuncu yok.
- Gruplar arasında küçük ama anlamlı bir dengesizlik var (p ≈ 0.009), sınırlama olarak not edildi.
- 49.854 round'luk tek kayıt veri hatası olarak çıkarılacak, diğer değerler kalacak.
- Hiç oynamayan oyuncular analizde kalacak.